In [6]:
# %pip install --upgrade pandas
%pip install roboflow ultralytics pandas matplotlib opencv-python-headless seaborn scikit-learn

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 1.9 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 3.5 MB/s eta 0:00:00a 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_PATH = '../weights/best.pt'  # Path to your trained YOLOv8 weights
TEST_IMAGES_DIR = '../eagle-eyes-23/test/images'          # Path to test images
OUTPUT_CSV = 'qc_validation_results.csv'          # Where to save detailed results

# SYSTEM THRESHOLDS (From Project Book)
EXPECTED_BLOCK_COUNT = 14
INTENSITY_THRESHOLD_FADING = 50   # Avg pixel intensity (0-255). >50 is Faded.
STD_DEV_THRESHOLD_SMUDGE = 25     # Pixel std deviation. >25 is Smudged.
LAPLACIAN_VAR_THRESHOLD = 100     # Texture focus. <100 is Unclear.
PIXEL_TO_MM_RATIO = 0.06 / 10     # Example calibration (Needs your actual calibration value!)
MISALIGNMENT_THRESHOLD_MM = 0.06 

# ==========================================
# 1. DEFINE LOGIC FUNCTIONS
# ==========================================

def check_print_quality(image, boxes):
    """
    Extracts Q-Block ROIs and checks for Fading and Smudging.
    Returns: status (str), value (float)
    """
    max_intensity = 0
    max_std_dev = 0
    
    for box in boxes:
        # Get coordinates (ensure they are within image bounds)
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(image.shape[1], x2), min(image.shape[0], y2)
        
        # Extract Region of Interest (ROI)
        roi = image[y1:y2, x1:x2]
        
        if roi.size == 0: continue

        # Convert to grayscale if not already
        if len(roi.shape) == 3:
            gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        else:
            gray_roi = roi

        # CHECK FADING (Average Intensity)
        avg_intensity = np.mean(gray_roi)
        if avg_intensity > max_intensity:
            max_intensity = avg_intensity

        # CHECK SMUDGING (Standard Deviation)
        std_dev = np.std(gray_roi)
        if std_dev > max_std_dev:
            max_std_dev = std_dev

    # Evaluate against thresholds
    if max_intensity > INTENSITY_THRESHOLD_FADING:
        return "FAIL_FADING", max_intensity
    
    if max_std_dev > STD_DEV_THRESHOLD_SMUDGE:
        return "FAIL_SMUDGING", max_std_dev

    return "PASS", max_intensity

def check_alignment(image_width, boxes):
    """
    Checks distance of Q-Blocks from image edge.
    """
    max_dist_mm = 0
    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # Calculate distance to nearest edge (simplified for left/right edges)
        dist_left = x1
        dist_right = image_width - x2
        min_dist_pixel = min(dist_left, dist_right)
        
        # Convert to mm
        dist_mm = min_dist_pixel * PIXEL_TO_MM_RATIO
        
        if dist_mm > max_dist_mm:
            max_dist_mm = dist_mm
            
    if max_dist_mm > MISALIGNMENT_THRESHOLD_MM:
        return "FAIL_ALIGNMENT", max_dist_mm
    
    return "PASS", max_dist_mm

# ==========================================
# 2. MAIN EVALUATION LOOP
# ==========================================

def run_evaluation():
    print(f"Loading model from {MODEL_PATH}...")
    model = YOLO(MODEL_PATH)
    
    results_data = []
    
    print(f"Starting inference on {TEST_IMAGES_DIR}...")
    image_files = [f for f in os.listdir(TEST_IMAGES_DIR) if f.lower().endswith(('.jpg', '.png', '.bmp'))]
    
    for img_file in image_files:
        img_path = os.path.join(TEST_IMAGES_DIR, img_file)
        
        # Load Image
        image = cv2.imread(img_path)
        if image is None: continue
        
        # ---------------------------
        # STAGE 1: VISION (YOLO)
        # ---------------------------
        results = model(img_path, verbose=False)[0]
        boxes = results.boxes
        detection_count = len(boxes)
        
        # Determine Ground Truth from filename (Assumes format like "ticket_NG_001.jpg")
        # ADJUST THIS LINE to match your actual filename labeling convention!
        ground_truth = "NOT_GOOD" if "NG" in img_file else "GOOD"
        
        # ---------------------------
        # STAGE 2: LOGIC CHECKS
        # ---------------------------
        final_prediction = "PASS"
        failure_reason = "NONE"
        
        # Check 1: Count
        if detection_count != EXPECTED_BLOCK_COUNT:
            final_prediction = "FAIL"
            failure_reason = "COUNT_MISMATCH"
        
        # Check 2: Print Quality (if count passed)
        if final_prediction == "PASS":
            quality_status, value = check_print_quality(image, boxes)
            if quality_status != "PASS":
                final_prediction = "FAIL"
                failure_reason = quality_status
                
        # Check 3: Alignment (if previous passed)
        if final_prediction == "PASS":
            align_status, val = check_alignment(image.shape[1], boxes)
            if align_status != "PASS":
                final_prediction = "FAIL"
                failure_reason = align_status

        # Map to Output Format
        system_output = "NOT_GOOD" if final_prediction == "FAIL" else "GOOD"
        
        results_data.append({
            "image": img_file,
            "ground_truth": ground_truth,
            "prediction": system_output,
            "failure_reason": failure_reason,
            "detections": detection_count
        })

    # ==========================================
    # 3. CALCULATE METRICS
    # ==========================================
    df = pd.DataFrame(results_data)
    
    # Save raw results
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\nProcessing complete. Results saved to {OUTPUT_CSV}")
    
    # Calculate Sklearn Metrics
    y_true = df['ground_truth']
    y_pred = df['prediction']
    
    print("\n" + "="*40)
    print("SYSTEM PERFORMANCE REPORT")
    print("="*40)
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label="NOT_GOOD")
    rec = recall_score(y_true, y_pred, pos_label="NOT_GOOD")
    f1 = f1_score(y_true, y_pred, pos_label="NOT_GOOD")
    
    print(f"Accuracy:  {acc:.2%}")
    print(f"Precision: {prec:.2%} (Reliability: When we say NG, is it really NG?)")
    print(f"Recall:    {rec:.2%} (Safety: Did we catch all NGs?)")
    print(f"F1 Score:  {f1:.4f}")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred, labels=["GOOD", "NOT_GOOD"]))
    
    print("\nDetailed Report:")
    print(classification_report(y_true, y_pred, target_names=["GOOD", "NOT_GOOD"]))

if __name__ == "__main__":
    run_evaluation()

Loading model from ../weights/best.pt...
Starting inference on ../eagle-eyes-23/test/images...

Processing complete. Results saved to qc_validation_results.csv

SYSTEM PERFORMANCE REPORT
Accuracy:  90.58%
Precision: 90.58% (Reliability: When we say NG, is it really NG?)
Recall:    100.00% (Safety: Did we catch all NGs?)
F1 Score:  0.9506

Confusion Matrix:
[[  0  13]
 [  0 125]]

Detailed Report:
              precision    recall  f1-score   support

        GOOD       0.00      0.00      0.00        13
    NOT_GOOD       0.91      1.00      0.95       125

    accuracy                           0.91       138
   macro avg       0.45      0.50      0.48       138
weighted avg       0.82      0.91      0.86       138



/Users/vishali/Documents/CDA-500/prototype/eagle_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/vishali/Documents/CDA-500/prototype/eagle_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/vishali/Documents/CDA-500/prototype/eagle_env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control thi